# Operaciones de aprendizaje automático
## Maestría en Inteligencia Artificial Aplicada

### TC5061 | Proyecto MiBici: propuesta y base reproducible | Colaborativo

#### **Equipo:** 7
#### **Nombre:** Luis Arturo Sánchez Davila
#### **Matrícula:** A01840576

## Objetivo
Notebook de **mejora del modelo de P3 (clustering)**: propone una alternativa al baseline (`0.04-lasd-baseline-p3`) —PCA sobre el perfil de uso y K-Means con más grupos—, la evalúa contra criterios de aceptación acordados antes de ver los resultados y registra el run en MLflow, aunque no se acepte.

## Entradas y salidas
- **Entrada:** `data/processed/viajes_modelado_AAAA-MM_AAAA-MM.parquet` (`0.03-lasd-limpieza-p3`). El baseline se recalcula con el mismo código y la misma semilla que en `0.04-lasd-baseline-p3`, así que no depende de que ese notebook haya corrido.
- **Salidas:** los archivos `reports/figures/p3_pca_*` y, si se pide, el run `pca_kmeans` en MLflow (experimento `mibici-equipo07` por defecto).

## Nota de reproducibilidad
El notebook se ejecuta desde la carpeta `notebooks/`. Las cifras corresponden a la corrida del 15-oct-2026 con los 81 meses del rango. La celda 4.9 solo escribe en el servidor de MLflow del equipo si `P3_REGISTRAR_MLFLOW=true`; por defecto no registra.

## Serie de notebooks de P3
| Notebook | Contenido | Entrega |
|---|---|---|
| `0.01-lasd-exploracion-carga` | Paso 1: carga y descripción | Dataset normalizado en `data/interim/` |
| `0.02-lasd-eda-p3` | Paso 2: análisis exploratorio | Hallazgos de calidad, tiempo y estaciones |
| `0.03-lasd-limpieza-p3` | Paso 3: limpieza, partición y preprocesamiento | `data/processed/` (viajes de modelado, estaciones y demanda) |
| `0.04-lasd-baseline-p3` | Paso 4 (4.1 a 4.4): modelo base y registro en MLflow | Modelo base y métricas (0.05 lo recalcula con el mismo código y semilla) |
| `0.05-lasd-mejoramodelo-p3` | Paso 4 (4.5 a 4.9): modelo alternativo y registro en MLflow | Comparación contra el baseline |

In [ ]:
# Importación de librerías y registro de versiones (evidencia de reproducibilidad)
from pathlib import Path
import sys

import matplotlib
import numpy as np
import pandas as pd
import sklearn

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)

## Configuración y entradas

Se carga el dataset de modelado y se recalcula el baseline con `train.p3_ajustar_baseline` (mismo código y semilla que `0.04-lasd-baseline-p3`; determinista, unos segundos). Todos los modelos se comparan en el espacio estandarizado de ese baseline (`z_train`).

La lógica reutilizable vive en el módulo `tc5061_proyecto_equipo07` (funciones `p3_*`); este notebook la llama y documenta los hallazgos.

In [ ]:
# Configuración compartida (el notebook se ejecuta desde notebooks/)
try:
    from tc5061_proyecto_equipo07 import dataset
except ModuleNotFoundError:  # paquete sin instalar; `make requirements` lo instala con `-e .`
    sys.path.insert(0, str(Path.cwd().parent))
    from tc5061_proyecto_equipo07 import dataset
from tc5061_proyecto_equipo07 import config as cfg
from tc5061_proyecto_equipo07 import plots
from tc5061_proyecto_equipo07.config import MES_FIN, MES_INICIO
from tc5061_proyecto_equipo07.modeling import train

# Opciones de visualización de pandas
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
pd.set_option("display.max_colwidth", None)  # no recortar textos largos

# Entrada: viajes de modelado y baseline (el mismo de 0.04: mismo código, misma semilla)
limpios = train.p3_cargar_modelado(MES_INICIO, MES_FIN)
nomenclatura = dataset.p3_cargar_estaciones()
base = train.p3_ajustar_baseline(limpios)
del limpios

K_ELEGIDO = base.k_elegido
metricas = base.metricas
COLUMNAS_MODELO = base.columnas_modelo
HORAS_SERVICIO = cfg.P3_HORAS_SERVICIO
columnas_lab = [c for c in COLUMNAS_MODELO if c.startswith("lab_")]
columnas_fin = [c for c in COLUMNAS_MODELO if c.startswith("fin_")]

print(f"Baseline: k = {K_ELEGIDO} | silueta {metricas['silueta']:.3f} | ARI {metricas['ari_valid']:.3f}")
print(f"Estaciones en entrenamiento: {len(base.x_train)} | comunes con validación: {len(base.comunes)}")

### 4.5 Modelo alternativo: PCA sobre el perfil horario + K-Means

El baseline mezcla 40 columnas de perfil horario con 4 indicadores escalares, y el perfil horario domina la distancia. Además, con `k = 2` solo separa "salidas de mañana" de "salidas de tarde". La alternativa mantiene las mismas variables y el mismo conjunto de estaciones, y cambia la representación y el número de grupos:

| Elemento | Decisión |
|---|---|
| **Datos y variables** | Los mismos del baseline (362 estaciones, 44 variables, solo entrenamiento) |
| **Representación** | Estandarización, **PCA** (3 a 10 componentes) y K-Means sobre los componentes |
| **Pesos por bloque** | Variante evaluada: cada bloque (laboral, fin de semana, indicadores) aporta la misma varianza total |
| **`k`** | De 3 a 6; no se fuerza `k = 2` |
| **Semillas** | Cada configuración se corre con 5 semillas y se compara el promedio; elegir con una sola semilla premia la suerte |
| **Comparación justa** | Silueta, Davies-Bouldin y Calinski-Harabasz se calculan en el **espacio estandarizado del baseline**, no en el espacio PCA, donde una dimensión menor "mejora" sola |

**Criterios de aceptación** (acordados antes de ver los resultados):
1. Silueta mayor que la del baseline (0.145).
2. ARI de estabilidad igual o mayor que el del baseline (0.66).
3. Tamaño mínimo de grupo de al menos 20 estaciones.
4. Cada grupo se puede nombrar con su perfil horario y su ubicación (se evalúa en 4.8).

Si no cumple, el resultado se documenta igual y el baseline sigue siendo el piso.

In [ ]:
# Referencia: el baseline en el mismo espacio estandarizado, para cada k y con la misma prueba de estabilidad
baseline_por_k = train.p3_baseline_por_k(base)
baseline_por_k.loc[2:6].round(3)

### 4.6 Búsqueda y selección

Se prueba con y sin pesos por bloque, 6 números de componentes y `k` de 3 a 6, con 5 semillas cada una (240 ajustes). La selección usa solo configuraciones que cumplen los criterios 2 y 3; entre ellas, la de mayor silueta promedio. Nótese que el ARI de estabilidad usa el año de validación para elegir, no el de prueba, que sigue sin tocarse.

In [ ]:
# Búsqueda (con y sin pesos por bloque, componentes de PCA y k; cada una con varias semillas) y selección
alt = train.p3_entrenar_alternativo(base)
elegido, k_alt = alt.elegido, alt.k_alt

alt.reserva.sort_values("silueta", ascending=False).head(8).round(3)

### 4.7 Evaluación contra el baseline

In [ ]:
# Comparación con el baseline y criterios de aceptación acordados (el log de la celda anterior trae el mismo veredicto)
for nombre, cumple in alt.criterios.items():
    print(("CUMPLE    " if cumple else "NO CUMPLE ") + nombre)
ACEPTADO = alt.aceptado
print("\nVeredicto cuantitativo:", "ACEPTADO" if ACEPTADO else "NO ACEPTADO: el baseline sigue siendo el piso")
print(f"Varianza explicada por los {int(elegido['componentes'])} componentes: {alt.varianza_pca:.1%}")
print(f"Silueta entre semillas: min {elegido['silueta_min']:.3f}, promedio {elegido['silueta']:.3f}")
print(f"ARI entre semillas: min {elegido['ari_min']:.3f}, promedio {elegido['ari_valid']:.3f}")
alt.comparacion.round(3)

**Observaciones de la evaluación:**
- **Selección:** de 48 configuraciones, 18 cumplen estabilidad y tamaño mínimo. Gana PCA **sin pesos por bloque**, 5 componentes (56.2 % de la varianza) y `k = 3`. Las mejores configuraciones ponderadas (silueta 0.116 y 0.104) quedan por debajo de las no ponderadas (0.137 a 0.140): igualar el peso de los bloques no ayudó en el espacio común.
- **Veredicto contra los criterios:** la silueta es **0.140 contra 0.145 y no cumple** (−0.005). El ARI de estabilidad (0.676 en promedio) y el tamaño mínimo de grupo (42) sí cumplen. Por el criterio 1, el resultado es **NO ACEPTADO** y el baseline sigue siendo el piso.
- **Sensibilidad a la semilla:** la silueta es estable (0.139 a 0.140), pero el ARI promedia 0.676 con un mínimo de 0.639 entre semillas (0.675 con la del proyecto), y el grupo más chico llega a 42 estaciones en el peor caso (68 con la del proyecto). La ventaja en estabilidad sobre el baseline (0.656) es pequeña y no concluyente. Por eso la selección usa el promedio de varias semillas y no una sola corrida.
- **A igual `k`, no hay una mejora clara:** con `k = 3` el baseline tiene silueta 0.139, Davies-Bouldin 2.265, ARI 0.689 y grupo mínimo de 68; el alternativo, 0.140, 2.223, 0.676 y 42. Mejora apenas las métricas internas, pero es menos estable y su grupo más chico es menor. Con 79 meses el alternativo sí superaba al baseline a igual `k` (ARI 0.698 contra 0.660); con los 81 meses esa ventaja desaparece, así que no era robusta a los datos.
- **Sobre el criterio 1:** compara un modelo de `k = 3` contra el de `k = 2` del baseline, y el baseline optimiza esa métrica en ese espacio; además la silueta cae al aumentar `k` (0.145, 0.139, 0.114). Si el equipo quiere `k ≥ 3` para decidir, el criterio justo es comparar a igual `k`, y con los datos actuales tampoco ahí hay ventaja clara. Esa decisión es del equipo; aquí se reporta con el criterio acordado.
- **Calidad absoluta:** una silueta de 0.14 sigue indicando estructura débil. Las estaciones forman más un continuo que grupos separados; ningún modelo de esta familia cambia eso.

### 4.8 Interpretación

Se compara el perfil horario, los indicadores y la ubicación de los tres grupos, y cómo se relacionan con los dos del baseline.

In [ ]:
# Interpretación: perfil horario, indicadores y ubicación de cada grupo del modelo alternativo
asignacion_alt = train.p3_asignacion_alternativo(alt, nomenclatura)
fig = plots.p3_grafica_perfiles_y_mapa(asignacion_alt, k_alt, columnas_lab, columnas_fin, HORAS_SERVICIO)

# Relación con los grupos del baseline: filas = alternativo, columnas = baseline
print("Estaciones por grupo alternativo (filas) y grupo del baseline (columnas):")
display(pd.crosstab(asignacion_alt["grupo_alt"], base.caracteristicas_train["grupo"]))
train.p3_resumir_alternativo(asignacion_alt)

**Observaciones de la interpretación:**
- **Grupo 2 (151 estaciones): salida de tarde y alto volumen.** Pico a las 18:00 (12.7 % de sus salidas laborales), 35.6 % de las salidas entre 17 y 20 h, mediana de 30.3 viajes por día (la más alta) y balance de −9.6 %: tiende a perder bicicletas. Son 150 de las 160 estaciones del grupo 1 del baseline.
- **Grupo 0 (143 estaciones): salida de mañana.** Pico a las 8:00 (11.7 %), 32.0 % de las salidas entre 6 y 9 h, 19.9 viajes por día y balance de +4.5 %: tiende a acumular bicicletas. Son 138 de las 202 estaciones del grupo 0 del baseline.
- **Grupo 1 (68 estaciones): perfil intermedio.** Es lo que el modelo alternativo aporta: parte del grupo 0 del baseline (63 estaciones) queda aparte, con salidas repartidas entre mañana (21.7 %) y tarde (30.1 %), duración mediana mayor (10.7 min contra 9.3 y 9.6) y proporción de viajes circulares más alta (8.3 %). Tiene 18.7 viajes por día y balance cercano a cero (−2.3 %).
- **Ubicación:** el grupo de salida de tarde (2) es el más cercano al centro de la red (distancia mediana de 2.2 km contra 3.0 km de los otros dos); la diferencia entre grupos es de 1 a 2 km, así que no hay una separación geográfica nítida y los grupos se distinguen sobre todo por uso.
- **Utilidad para decidir:** el grupo 1 es una hipótesis útil (estaciones de uso mixto o de paso que conviene revisar aparte), pero con una silueta de 0.14 sus límites son difusos y la partición cambia entre semillas. Los tres grupos se pueden nombrar, por lo que el criterio 4 se cumple de forma cualitativa.

### 4.9 Registro en MLflow

El modelo alternativo se registra en el mismo experimento que el baseline (`mibici-equipo07` por defecto) con el run `pca_kmeans`, o `<MLFLOW_RUN_NAME>-pca_kmeans`, sin tocar el run `baseline`. Solo se registra si se pide con `P3_REGISTRAR_MLFLOW=true`; la URL sale de `MLFLOW_TRACKING_URI`.
- **Etiquetas:** incluyen `etapa = alternativo`, `aceptado` (según los criterios de 4.7) y `referencia = baseline`, para filtrar en la interfaz.
- **Métricas:** silueta (promedio entre semillas y con la semilla del proyecto), Davies-Bouldin, Calinski-Harabasz, ARI de estabilidad, tamaño mínimo, varianza del PCA y diferencias contra el baseline.
- **Artefactos:** asignación por estación, resumen de grupos, búsqueda completa, comparación y la figura de perfiles y mapa, en la carpeta `p3_pca`.

La celda es idempotente, igual que la del baseline: si ya existe un run terminado con ese nombre, no crea otro (`P3_REGISTRAR_DE_NUEVO=true` lo fuerza). Los artefactos también quedan en `reports/figures/`, se registre o no. Se registra aunque el modelo no se acepte, porque un resultado negativo también es evidencia.

In [ ]:
# Registro del modelo alternativo en MLflow (train.p3_registrar_alternativo; URL, experimento y nombre vienen del entorno)
REGISTRAR_ALT_EN_MLFLOW = cfg.P3_REGISTRAR_MLFLOW   # False por defecto; P3_REGISTRAR_MLFLOW=true para registrar
REGISTRAR_ALT_DE_NUEVO = cfg.P3_REGISTRAR_DE_NUEVO

if REGISTRAR_ALT_EN_MLFLOW:
    train.p3_registrar_alternativo(alt, nomenclatura, REGISTRAR_ALT_DE_NUEVO, origen="0.05-lasd-mejoramodelo-p3")
else:
    rutas = train.p3_guardar_artefactos_alternativo(alt, nomenclatura)
    print("Registro en MLflow desactivado (P3_REGISTRAR_MLFLOW=false). Artefactos locales:", [r.name for r in rutas])